In [0]:

%run ./Includes/Copy-Datasets

In [0]:
dbutils.widgets.text("dataset_bookstore", dataset_bookstore)

In [0]:
%sql
CREATE OR REPLACE TABLE orders AS
SELECT * FROM parquet.`${dataset_bookstore}/orders`

In [0]:
%sql
SELECT * from orders

In [0]:
%sql
DESCRIBE HISTORY orders

In [0]:
%sql
INSERT OVERWRITE orders
SELECT * FROM parquet.`${dataset_bookstore}/orders`

In [0]:
%sql
DESCRIBE HISTORY orders

In [0]:
%sql
INSERT INTO orders
SELECT * FROM parquet.`${dataset_bookstore}/orders-new`

In [0]:
%sql
SELECT * FROM parquet.`${dataset_bookstore}/orders-new`
    

In [0]:
%sql
DESCRIBE HISTORY orders

In [0]:
%sql
CREATE OR REPLACE TABLE customers AS
SELECT * FROM json.`${dataset_bookstore}/customers-json`

In [0]:
%sql
SELECT * from customers

In [0]:
%sql
SELECT count(*) FROM customers

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_updates AS
SELECT * FROM json.`${dataset_bookstore}/customers-json-new`


In [0]:
%sql
SELECT * FROM customers_updates

In [0]:
%sql
MERGE INTO customers c
USING customers_updates cus
ON c.customer_id = cus.customer_id
WHEN MATCHED AND c.email IS NULL AND cus.email IS NOT NULL THEN UPDATE SET c.email = cus.email
,c.updated = cus.updated WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW books_updates
(book_id STRING , title STRING, author STRING, category STRING, price DOUBLE)
USING csv
OPTIONS (path "${dataset_bookstore}/books-csv-new", header "true", delimiter ";");

SELECT * FROM books_updates

In [0]:
%sql

MERGE INTO books b
USING books_updates bu
ON b.book_id = bu.book_id
WHEN NOT MATCHED AND b.category = 'Computer Science' THEN INSERT *